# Setup

## Imports

In [ ]:
# matplotlib
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# others
import airr
import colorspacious as cs
import numpy as np
import seaborn as sns

# add repertoire_zoo
import repertoire_zoo.bird as br
import repertoire_zoo.giraffe as gr

## Setup directories - Read in data

In [ ]:
# Define project/job uuid
project_id = ... # project uuid
job_id = ... # jov uuid

# pull out repertoires and names
groups_of_interest = [ ... ] # insert groups to compare (to compare all replace `...` with `None`)

# identify control group(s) to compare against
control_group_names = [ ... ] # insert control groups (for no control replace `...` with `None`)

# Define data directories
data_dir = ... #/path/to/data/directory/
repcalc_dir = data_dir + 'vdjserver/'+job_id+'/'
analysis_dir = data_dir + 'analysis/'
fig_dir = analysis_dir + 'figures/'

# Read in repertoire and repertoire group info
# The default name (repertoires.airr.json) is being used, but custom airr.json files can
# be used for defining repertoires and repertoire groups. 
repertoires_airr_json_filename = 'repertoires.airr.json'
sample_info_df = br.read_repertoire_info_airr(repcalc_dir + repertoires_airr_json_filename)
repertoire_group_df = br.read_repertoire_group_info_airr(repcalc_dir + repertoires_airr_json_filename)
data = airr.read_airr(repcalc_dir + repertoires_airr_json_filename)

if groups_of_interest == [None]:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] }
else:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] if obj['repertoire_group_name'] in groups_of_interest }
repertoires = { obj['repertoire_id'] : obj for obj in data['Repertoire']}

order_map = {name: idx for idx, name in enumerate(groups_of_interest)}
repertoire_groups = dict(sorted(
    repertoire_groups_temp.items(),
    key=lambda item: order_map.get(item[1]['repertoire_group_name'], len(groups_of_interest))
))

# make groups tuple
groups_tuple = [
    (
        repertoire_groups[rep_group_id]['repertoire_group_id'],
        repertoire_groups[rep_group_id]['repertoire_group_name'],
        [reps.get('repertoire_id') for reps in repertoire_groups[rep_group_id]['repertoires']]
    ) for rep_group_id in repertoire_groups
]

for i, rep_group_id in enumerate(repertoire_groups):
    print(rep_group_id, i, repertoire_groups[rep_group_id]['repertoire_group_name'])

condition_counts = {e[1] : e[2] for e in groups_tuple}

## Colormaps

In [ ]:
# set hue order (use condition names)
# By default the hue_order will be the same as the groups of interest. 
# If this is different, redfine `hue_order` as a list of groups from `groups_of_interest`
hue_order = groups_of_interest

cmap = mpl.colormaps['nipy_spectral'] # gist_rainbow
cmin = 0.0
cmax = 1.0

def generate_colormap(base_colors, all_names=None, n_shades=7, min_light=0, max_light=0.7):
    """
    Generate a dictionary of colors where each base color has n_shades variants
    going from dark to light, excluding white.

    Parameters
    ----------
    base_colors : dict
        Mapping of base condition name to base color (e.g., {'A': 'red', 'B': 'blue'}).
    n_shades : int
        Number of shades per base color.
    min_light : float
        Minimum lightness factor (0 = black, 1 = white).
    max_light : float
        Maximum lightness factor (0 = black, 1 = white).

    Returns
    -------
    dict : {condition_name: hex_color}
    """
    cmap_dict = {}

    for i, base in enumerate(base_colors):
        base_rgb = np.array(mcolors.to_rgb(base))
        for ii, factor in enumerate(np.linspace(min_light, max_light, n_shades)):
            # base_rgb * (1-factor) -> responsible for color being removed
            # np.ones(3) * factor -> responsible for the color interpolating to white
            shade_rgb = base_rgb * (1-factor) + (np.ones(3) * factor)
            
            cmap_dict[f"{all_names[i*n_shades+ii]}"] = mcolors.to_hex(shade_rgb)

    return cmap_dict

def show_colormap(color_cmap, ax=None, xtick_labels=True, xtick_height=-0.3):
    """
    Display colors associated in a color_cmap dictionary.

    Parameters
    ----------
    color_cmap : dict
        Dictionary mapping labels to color values (e.g., {'A': 'red', 'B': '#00FF00'}).
    """
    labels = list(color_cmap.keys())
    colors = list(color_cmap.values())

    # Create a figure
    if ax is None:
        fig, ax = plt.subplots(figsize=(len(labels), 1))
    else:
        fig = ax.get_figure()

    for i, color in enumerate(colors):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
        if xtick_labels: ax.text(i + 0.5, xtick_height, labels[i], ha='center', va='center', fontsize=10, rotation=90)

    ax.set_xlim(0, len(colors))
    ax.set_ylim(0, 1)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xticks([])
    ax.set_yticks([])

    return fig, ax


# V
v_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='gene'
)['gene'].unique()
v_genes_cmap = {
    v_gene : cmap(np.linspace(cmin,cmax,len(v_genes))[i])
    for i, v_gene in enumerate(v_genes)
}

v_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='subgroup'
)['gene'].unique()
v_subgroup_cmap = {
    v_subgroup : cmap(np.linspace(cmin,cmax,len(v_subgroups))[i])
    for i, v_subgroup in enumerate(v_subgroups)
}

# J
j_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='gene'
)['gene'].unique()
j_genes_cmap = {
    j_gene : cmap(np.linspace(cmin,cmax,len(j_genes))[i])
    for i, j_gene in enumerate(j_genes)
}

j_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='subgroup'
)['gene'].unique()
j_subgroup_cmap = {
    j_subgroup : cmap(np.linspace(cmin,cmax,len(j_subgroups))[i])
    for i, j_subgroup in enumerate(j_subgroups)
}

## cond_cmap
conds = br.load_and_prepare_data_junction_aa(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='.igblast.makedb.allele.clone.group.',
    call_type='junction_aa_length'
)['condition'].unique()

# Colorblind safe colors and colorblind analysis
# number of colors = number of base groups to compare
base_colors = ['#A577F9', '#699B59', '#D36135', '#086788', '#FFBC42']
cond_cmap = {
    cond : base_colors[i]
    for i, cond in enumerate(conds)
}

names = [cond[1] for cond in groups_tuple]

deuteranomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "deuteranomaly",
        "severity": 100
    },
    end = "sRGB1"
)

protanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "protanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

tritanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "tritanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

fig, axes = plt.subplots(4, 1, figsize=(len(list(cond_cmap.keys())), 4))
colorblind_simulated_cmaps = [deuteranomaly_simulated_cmap, protanomaly_simulated_cmap, tritanomaly_simulated_cmap]
gr.show_colormap(cond_cmap, axes[0], False)
for i, (ax, colorblind_simulated_cmap) in enumerate(zip(axes[1:], colorblind_simulated_cmaps)):
    colorblind_simulated_cmap = [np.clip(e, 0, 1) for e in colorblind_simulated_cmap]
    colorblind_simulated_cmap = {k:v for k,v in zip(cond_cmap.keys(), colorblind_simulated_cmap)}
    if i != 2: gr.show_colormap(colorblind_simulated_cmap, ax, False)
    else: gr.show_colormap(colorblind_simulated_cmap, ax, True, 2.5)

axes[0].set_ylabel('Normal', rotation=0, labelpad=20)
axes[1].set_ylabel('Deuteranomaly', rotation=0, labelpad=39)
axes[2].set_ylabel('Protanomaly', rotation=0, labelpad=32)
axes[3].set_ylabel('Tritanomaly', rotation=0, labelpad=32)
fig.suptitle('Colorblind Analysis', fontsize=24, y=1.02)
fig.show()

# V

## Bar graphs of V family frequency

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'v_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage=processing_stage,
    call_type=call_type,
    level=level,
    threshold=threshold
)

datapoints = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage=processing_stage,
    call_type=call_type,
    level=level,
    threshold=threshold,
    datapoints=True,
    categories=list(df['gene'].unique())
)

fig, ax = gr.plot_duplicate_frequency(
    df=df,
    datapoints=datapoints,
    hue_order=hue_order,
    palette=cond_cmap,
    title={'t':'VH Gene Family Duplicate Frequency', 'fontsize':20},
    ylim=ylim,
    errorbar='se',
    split=True,
    legend=True
)

fig.show()

## Heat maps

### Raw data

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'v_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make figure
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))
sns.heatmap(matrix, ax=ax, annot=True)
fig.show()

### Absolute difference per family (unpaired axis)

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'v_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make sure matrix is numeric
matrix = matrix.astype(float)

cmaps = []
ims = []
norms = []
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))

for j, col in enumerate(matrix.columns):
    vals = matrix[col].values  # shape (n_rows,)

    # set up colormap and norm
    cmap = LinearSegmentedColormap.from_list(col, ['Blue', 'White', 'Red'])
    cmaps.append(cmap)
    v_min = vals.min()
    v_center = vals[0]
    v_max = vals.max()
    # add jitter so that center value does not equal min or max for colormap
    if v_min == v_center:
        v_min -= 0.0000001
    if v_max == v_center:
        v_max += 0.0000001
    norm = mpl.colors.TwoSlopeNorm(vmin=v_min, vcenter=v_center, vmax=v_max)
    norms.append(norm)


    # draw the column as an image of shape (n_rows, 1)
    im = ax.imshow(vals[:, None], cmap=cmap, norm=norm, aspect='auto',
                extent=[j, j+1, 0, n_rows], origin='lower')
    ims.append(im)

    # add text annotations
    for k, v in enumerate(vals):
        ax.text(j + 0.5, k + 0.5, f"{v:.5f}", ha='center', va='center', fontsize=8, color='k')

# ticks and limits
ax.set_xticks(np.arange(n_cols) + 0.5)
ax.set_xticklabels(matrix.columns, rotation=90, y=-0.025)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_yticklabels(matrix.index)
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)

# add legend below each column
for j, (cmap, norm) in enumerate(zip(cmaps, norms)):
    cax = inset_axes(ax, width="80%", height="25%", loc="lower center",
                    bbox_to_anchor=(j, -0.45, 1, 1),  # shift horizontally
                    bbox_transform=ax.transData, borderpad=0)
    fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
                cax=cax, orientation="horizontal")
    cax.xaxis.set_ticks_position("bottom")   
    cax.xaxis.set_tick_params(labelrotation=45, labelsize=6)   

plt.show()


### Normalized difference

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'v_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make sure matrix is numeric
matrix = matrix.astype(float)

# make normalized matrix
norm_mat = gr.norm_cols(matrix, 'difference')

# make annotation matrix
annot_mat = gr.make_annotation_mat(matrix, norm_mat, 4, 4)

# make figure
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))

sns.heatmap(
    norm_mat,
    annot=annot_mat,
    fmt='',
    cmap="RdBu_r",
    vmin=-0.11,
    vmax=0.11,
    ax=ax
)

ax.set_title(
    'Difference Between Repertoire Group and Control by Gene Family',
    fontdict={'fontsize':20},
    x=(1.2-0.4)/2
)
ax.set_ylabel('Repertoire Group')
ax.set_xlabel('Gene Family')

fig.show()

# J

## Bar graph of J family frequency

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'j_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage=processing_stage,
    call_type=call_type,
    level=level,
    threshold=threshold
)

datapoints = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage=processing_stage,
    call_type=call_type,
    level=level,
    threshold=threshold,
    datapoints=True,
    categories=list(df['gene'].unique())
)

fig, ax = gr.plot_duplicate_frequency(
    df=df,
    datapoints=datapoints,
    hue_order=hue_order,
    palette=cond_cmap,
    title={'t':'JH Gene Family Duplicate Frequency', 'fontsize':20},
    ylim=ylim,
    errorbar='se',
    split=True,
    legend=True
)

fig.show()

## Heat maps

### Raw data

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'j_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make figure
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))
sns.heatmap(matrix, ax=ax, annot=True)
fig.show()

### Absolute difference per family (unpaired axis)

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'j_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make sure matrix is numeric
matrix = matrix.astype(float)

cmaps = []
ims = []
norms = []
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))

for j, col in enumerate(matrix.columns):
    vals = matrix[col].values  # shape (n_rows,)

    # set up colormap and norm
    cmap = LinearSegmentedColormap.from_list(col, ['Blue', 'White', 'Red'])
    cmaps.append(cmap)
    v_min = vals.min()
    v_center = vals[0]
    v_max = vals.max()
    # add jitter so that center value does not equal min or max for colormap
    if v_min == v_center:
        v_min -= 0.0000001
    if v_max == v_center:
        v_max += 0.0000001
    norm = mpl.colors.TwoSlopeNorm(vmin=v_min, vcenter=v_center, vmax=v_max)
    norms.append(norm)


    # draw the column as an image of shape (n_rows, 1)
    im = ax.imshow(vals[:, None], cmap=cmap, norm=norm, aspect='auto',
                extent=[j, j+1, 0, n_rows], origin='lower')
    ims.append(im)

    # add text annotations
    for k, v in enumerate(vals):
        ax.text(j + 0.5, k + 0.5, f"{v:.5f}", ha='center', va='center', fontsize=8, color='k')

# ticks and limits
ax.set_xticks(np.arange(n_cols) + 0.5)
ax.set_xticklabels(matrix.columns, rotation=90, y=-0.025)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_yticklabels(matrix.index)
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)

# add legend below each column
for j, (cmap, norm) in enumerate(zip(cmaps, norms)):
    cax = inset_axes(ax, width="80%", height="25%", loc="lower center",
                    bbox_to_anchor=(j, -0.45, 1, 1),  # shift horizontally
                    bbox_transform=ax.transData, borderpad=0)
    fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
                cax=cax, orientation="horizontal")
    cax.xaxis.set_ticks_position("bottom")   
    cax.xaxis.set_tick_params(labelrotation=45, labelsize=6)   

# plt.tight_layout()
plt.show()


### Normalized difference

In [ ]:
global threshold, ylim, processing_stage, call_type, level
threshold = 0.001 #0.02
ylim = (0,1)
processing_stage = 'igblast.makedb.gene.clone'
call_type = 'j_call'
level = 'subgroup'

df = br.load_gene_usage_group_data(
        repcalc_dir=repcalc_dir,
        groups=groups_tuple,
        processing_stage=processing_stage,
        call_type=call_type,
        level=level,
        threshold=threshold
    )

if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(df['condition'].unique())

matrix = df.pivot(
    index="condition",
    columns="gene",
    values="duplicate_frequency_avg"
).reindex(idx)

# make sure matrix is numeric
matrix = matrix.astype(float)

# make normalized matrix
norm_mat = gr.norm_cols(matrix, 'difference')

# make annotation matrix
annot_mat = gr.make_annotation_mat(matrix, norm_mat, 4, 4)

# make figure
n_rows, n_cols = matrix.shape
fig, ax = plt.subplots(figsize=(1.4*n_cols, 0.5*n_rows+1))

sns.heatmap(
    norm_mat,
    annot=annot_mat,
    fmt='',
    cmap="RdBu_r",
    vmin=-0.11,
    vmax=0.11,
    ax=ax
)

ax.set_title(
    'Difference Between Repertoire Group and Control by Gene Family',
    fontdict={'fontsize':20},
    x=(1.2-0.4)/2
)
ax.set_ylabel('Repertoire Group')
ax.set_xlabel('Gene Family')

fig.show()

# VJ Combo

## Chord diagram of VJ gene pairing

In [ ]:
nrows = 1
ncols = 3
figsize = (20,8) # overall figure size, will need to adjust as well as nrows and ncols scales.

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize,
    subplot_kw={'projection':'polar'})

max_y_lim=0
for i, group in enumerate(groups_tuple):
    if type(axes) == np.ndarray:
        if i==0:
            axes = axes.flatten()
        ax = axes[i]
    else:
        ax = axes
    gr.plot_vj_chord_diagram(
        df=br.load_and_prepare_data_vj_combo(
            repcalc_dir=repcalc_dir,
            group_id=group[0],
            combo_type='vj_combo',
            processing_stage='.igblast.makedb.gene.clone.group.'
        ),
        freq_col='duplicate_frequency_avg',
        ticks_interval=0.02,
        max_links=None,
        cmap=v_subgroup_cmap | j_subgroup_cmap,
        title={'t':f'{group[1]}, N={len(group[2])}', 'fontsie':16, 'y':1.08},
        ax=ax
    )
    if max_y_lim < ax.get_ylim()[1]:
        max_y_lim = ax.get_ylim()[1]

fontsize=int(max(24, 1.25 * fig.get_size_inches()[0]))

fig.suptitle(**{'t':'Chord Diagram of VH:JH gene pairing', 'y':0.98, 'fontsize':fontsize})
plt.tight_layout()
fig.show()